# AH | 02. ETL diario, local o remoto

**Objetivo:** transformar los reportes TXT en tablas limpias e incrementales para Power BI.

- Cada fila del TXT representa **una visita**, no una apertura.

- Se conserva el histórico y los `IdVisita` de cargas anteriores.

- **MODO = "L"**: archivos de `input`. **MODO = "R"**: descarga desde SFTP con la misma transformación.

## 01. Importar bibliotecas
`paramiko` solo se utiliza en modo remoto (`R`).

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import posixpath
import re
import pandas as pd

## 02. Carpeta raíz
La carpeta raíz se obtiene del directorio de ejecución; ejecutar desde la raíz del proyecto o desde `notebooks/`.

In [ ]:
BASE = Path.cwd().resolve()
if BASE.name.lower() == "notebooks":
    BASE = BASE.parent
INPUT = BASE / "input"
CATALOGOS = BASE / "Catalogos"
OUTPUT = BASE / "output"
FECHA_PROCESO = pd.Timestamp.today().normalize()
assert INPUT.is_dir() and CATALOGOS.is_dir(), "Inicia Jupyter desde la raiz del proyecto o desde notebooks/."

## 03. Elegir modo de ejecución
**Única celda que necesitas cambiar** para elegir de dónde salen los reportes.

In [ ]:
MODO = "L"   # "L": local | "R": servidor SFTP
MODO = MODO.upper().strip()
if MODO not in ("L", "R"):
    raise ValueError('MODO solo admite "L" o "R"')
print("Modo seleccionado:", MODO)

## 04. Configuración del servidor SFTP
Los siguientes parámetros se utilizan únicamente en `MODO = "R"`: servidor, usuario, contraseña y carpeta de origen. Para el modo local (`MODO = "L"`), el proceso toma los archivos de `input/`.

In [ ]:
SFTP_HOST = "sftp.example.invalid"                # Ejemplo para modo R; configurar según el servidor
SFTP_USUARIO = "usuario_demo"               # Ejemplo para modo R; configurar según el servidor
SFTP_PASSWORD = ""                          # Completar solo al usar modo R
SFTP_CARPETA = "/ruta/remota/archivosVisitas"

## 05. Columnas esperadas
Se permite que cambie únicamente el encabezado de la segunda columna.

In [ ]:
COLUMNAS = ["email", "CampoExtra", "Badmail", "Baja", "Fecha envio",
            "Fecha open", "Opens", "Opens virales", "Fecha click",
            "Clicks", "Clicks virales", "Links", "IPs", "Navegadores", "Plataformas"]
NUMERICAS = ["Opens", "Opens virales", "Clicks", "Clicks virales"]
FINAL = COLUMNAS + ["ArchivoOrigen", "FechaVisita_Texto", "OrigenFechaVisita",
                   "FechaVisita", "VisitasRegistro", "EmailNormalizado",
                   "NavegadorAnalitico", "PlataformaAnalitica", "EstadoBadmail",
                   "EstadoBaja", "IdVisita"]

## 06. Comprobar la configuración remota
En modo local se omite este paso. En remoto se utilizan únicamente los cuatro datos anteriores.

In [ ]:
if MODO == "R":
    import paramiko
    if not all([SFTP_HOST, SFTP_USUARIO, SFTP_PASSWORD, SFTP_CARPETA]):
        raise ValueError("Completa los cuatro datos SFTP antes de usar R")

## 07. Abrir conexión SFTP
Se utiliza el puerto estándar 22 y se comprueba automáticamente la identidad del servidor mediante SSH. Si la conexión o autenticación falla, se detiene la descarga.

In [ ]:
def abrir_sftp():
    ssh = paramiko.SSHClient()
    ssh.load_system_host_keys()
    ssh.set_missing_host_key_policy(paramiko.RejectPolicy())
    try:
        ssh.connect(SFTP_HOST, port=22, username=SFTP_USUARIO,
                    password=SFTP_PASSWORD, timeout=20,
                    look_for_keys=False, allow_agent=False)
        return ssh, ssh.open_sftp()
    except Exception:
        ssh.close()
        raise

## 08. Seleccionar origen
El modo remoto usa un subdirectorio temporal del área `input`, sin mezclar archivos descargados con los originales locales.

In [ ]:
FUENTE = INPUT if MODO == "L" else INPUT / "_sftp_cache"
FUENTE.mkdir(parents=True, exist_ok=True)
PATRON = re.compile(r"report_\d+\.txt$")
archivos = []

## 09. Descargar los archivos del SFTP cuando corresponda
Solo archivos `report_N.txt`; se descarga primero a `.part` para evitar tratar una transferencia incompleta como un reporte. En local este paso no hace nada.

In [ ]:
if MODO == "R":
    ssh, sftp = abrir_sftp()
    try:
        nombres = [a.filename for a in sftp.listdir_attr(SFTP_CARPETA)
                   if PATRON.fullmatch(a.filename)]
        for nombre in nombres:
            temporal = FUENTE / (nombre + ".part")
            try:
                sftp.get(posixpath.join(SFTP_CARPETA, nombre), str(temporal))
                os.replace(str(temporal), str(FUENTE / nombre))
            finally:
                if temporal.exists(): temporal.unlink()
    finally:
        sftp.close()
        ssh.close()

## 10. Ordenar los reportes
El mismo criterio se usa en ambos modos. Así `report_10` queda después de `report_9`.

In [ ]:
if MODO == "L":
    archivos = [a for a in FUENTE.glob("report_*.txt") if PATRON.fullmatch(a.name)]
else:
    archivos = [FUENTE / nombre for nombre in nombres]
archivos = sorted(archivos, key=lambda a: int(a.stem.split("_")[1]))
if not archivos:
    raise FileNotFoundError("No se encontraron reportes report_N.txt")
pd.DataFrame({"Archivo": [a.name for a in archivos]})

## 11. Ayudas de limpieza
Se conserva el texto original en las columnas recibidas; aquí solo normalizamos cuando hace falta analizarlo.

In [ ]:
def texto(x):
    return "" if pd.isna(x) else str(x).strip()

def vacio(x):
    return texto(x) in ("", "-")

def fecha(x):
    return pd.to_datetime(texto(x), dayfirst=True, errors="coerce") if not vacio(x) else pd.NaT

## 12. Leer y revisar la estructura
Los archivos de muestra presentan una segunda cabecera variable. Cualquier otro cambio detiene la ejecución.

In [ ]:
def leer(archivo):
    try:
        df = pd.read_csv(archivo, sep=",", encoding="utf-8-sig", dtype=str,
                         keep_default_na=False)
    except UnicodeDecodeError:
        df = pd.read_csv(archivo, sep=",", encoding="cp1252", dtype=str,
                         keep_default_na=False)
    cabeceras = list(df.columns)
    if len(cabeceras) != 15 or cabeceras[:1] + cabeceras[2:] != COLUMNAS[:1] + COLUMNAS[2:]:
        raise ValueError("Encabezados inesperados en " + archivo.name)
    df.columns = COLUMNAS
    df["ArchivoOrigen"] = archivo.name
    return df

## 13. Localizar el historial incremental
Para evitar duplicaciones, `archivos_procesados.json` debe permanecer junto a `CONSOLIDADO.csv`.

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
RUTA_ESTADO = OUTPUT / "archivos_procesados.json"
RUTA_CSV = OUTPUT / "CONSOLIDADO.csv"
RUTA_PENDIENTE = OUTPUT / ".carga_en_curso"
if RUTA_PENDIENTE.exists():
    raise RuntimeError("Carga anterior interrumpida; revisar output antes de continuar")
if RUTA_ESTADO.exists() != RUTA_CSV.exists():
    raise RuntimeError("Faltan el historial o el CONSOLIDADO; restaurar juntos")

## 14. Recuperar las huellas de archivos procesados
La firma SHA-256 permite detectar un archivo ya cargado o uno modificado con el mismo nombre.

In [ ]:
estado = json.loads(RUTA_ESTADO.read_text(encoding="utf-8")) if RUTA_ESTADO.exists() else {}
if not isinstance(estado, dict):
    raise ValueError("Historial de cargas inválido")
estado_nuevo = dict(estado)
print("Archivos en el historial:", len(estado))

## 15. Identificar archivos nuevos
Un mismo nombre con contenido cambiado produce error. Un mismo contenido con otro nombre se ignora para no duplicar visitas.

In [ ]:
nuevos, omitidos = [], []
for archivo in archivos:
    huella = hashlib.sha256(archivo.read_bytes()).hexdigest()
    if archivo.name in estado and estado[archivo.name] != huella:
        raise ValueError("Archivo previamente cargado modificado: " + archivo.name)
    if huella in estado_nuevo.values():
        omitidos.append(archivo.name)
    else:
        nuevos.append(archivo)
        estado_nuevo[archivo.name] = huella
print("Nuevos:", len(nuevos), "| Omitidos:", len(omitidos))

## 16. Leer los datos realmente nuevos
No se transforman ni duplican las filas que ya están en el histórico.

In [ ]:
partes = [leer(a) for a in nuevos]
datos = (pd.concat(partes, ignore_index=True) if partes
         else pd.DataFrame(columns=COLUMNAS + ["ArchivoOrigen"]))
print("Filas nuevas:", len(datos))

## 17. Convertir contadores
Un valor no numérico o negativo detiene la carga antes de exportar.

In [ ]:
for col in NUMERICAS:
    datos[col] = pd.to_numeric(datos[col], errors="raise").astype("int64")
    if (datos[col] < 0).any():
        raise ValueError("Contador negativo en " + col)
datos[["Opens", "Clicks"]].sum()

## 18. Definir la fecha analítica
Prioridad de fecha: apertura → envío → día del procesamiento, sin perder las fechas originales.

In [ ]:
fecha_open = pd.to_datetime(datos["Fecha open"].map(fecha), errors="coerce")
fecha_envio = pd.to_datetime(datos["Fecha envio"].map(fecha), errors="coerce")
datos["FechaVisita"] = fecha_open.fillna(fecha_envio).fillna(FECHA_PROCESO)
datos["OrigenFechaVisita"] = "PROCESO"
datos.loc[fecha_envio.notna(), "OrigenFechaVisita"] = "ENVIO"
datos.loc[fecha_open.notna(), "OrigenFechaVisita"] = "OPEN"
datos["OrigenFechaVisita"].value_counts()

## 18b. Detectar fechas con formato inválido
Una fecha escrita, pero imposible de interpretar, debe reportarse. No se convierte silenciosamente en fecha de proceso.

In [ ]:
mal_open = (~datos["Fecha open"].map(vacio)) & fecha_open.isna()
mal_envio = (~datos["Fecha envio"].map(vacio)) & fecha_envio.isna()
if mal_open.any() or mal_envio.any():
    raise ValueError("Fechas no interpretables: revisar los archivos nuevos")
print("Fechas originales interpretables: OK")

## 19. Completar campos de consumo
Una visita sigue siendo una fila, aunque tenga cero o varias aperturas.

In [ ]:
datos["FechaVisita_Texto"] = datos["FechaVisita"].dt.strftime("%d/%m/%Y %H:%M")
datos["VisitasRegistro"] = 1
datos["EmailNormalizado"] = datos["email"].str.strip().str.lower()
datos[["EmailNormalizado", "FechaVisita_Texto"]].head()

## 20. Interpretar navegador y plataforma
Se distingue un campo ausente sin apertura de una tecnología no identificada.

In [ ]:
def navegador(fila):
    valor = texto(fila["Navegadores"])
    if not vacio(valor): return valor
    return "Sin apertura registrada" if fila["Opens"] == 0 else "Navegador no identificado"

def plataforma(fila):
    valor = texto(fila["Plataformas"])
    if not vacio(valor) and valor.lower() != "unknown": return valor
    return "Sin apertura registrada" if vacio(valor) and fila["Opens"] == 0 else "Plataforma no identificada"

## 21. Etiquetar calidad sin alterar las fuentes
`Navegadores` y `Plataformas` originales permanecen intactos.

In [ ]:
datos["NavegadorAnalitico"] = datos.apply(navegador, axis=1)
datos["PlataformaAnalitica"] = datos.apply(plataforma, axis=1)
datos["EstadoBadmail"] = datos["Badmail"].map(
    lambda x: texto(x) if not vacio(x) else "Sin indicador registrado")
datos["EstadoBaja"] = datos["Baja"].map(
    lambda x: texto(x) if not vacio(x) else "Sin baja registrada")
datos[["NavegadorAnalitico", "PlataformaAnalitica"]].head()

## 22. Leer las visitas históricas
El identificador anterior siempre permanece estable.

In [ ]:
if RUTA_CSV.exists():
    anterior = pd.read_csv(RUTA_CSV, dtype=str, keep_default_na=False)
    if list(anterior.columns) != FINAL:
        raise ValueError("Estructura histórica incorrecta")
else:
    anterior = pd.DataFrame(columns=FINAL)
ultimo_id = int(pd.to_numeric(anterior["IdVisita"]).max()) if len(anterior) else 0
print("Visitas anteriores:", len(anterior), "| Último ID:", ultimo_id)

## 23. Añadir visitas nuevas
IDs secuenciales; no se reemplazan las visitas previamente exportadas.

In [ ]:
datos["IdVisita"] = range(ultimo_id + 1, ultimo_id + len(datos) + 1)
visitas = pd.concat([anterior, datos[FINAL]], ignore_index=True)
visitas["FechaVisita"] = pd.to_datetime(visitas["FechaVisita"], dayfirst=True)
visitas["IdVisita"] = visitas["IdVisita"].astype(int)
for col in NUMERICAS + ["VisitasRegistro"]:
    visitas[col] = visitas[col].astype(int)
print("Visitas acumuladas:", len(visitas))

## 24. Construir asociaciones de plataforma
Cada elemento reconocido se convierte en una relación única `(IdVisita, plataforma)` para RLS.

In [ ]:
filas = []
for fila in visitas.itertuples(index=False):
    for token in texto(fila.Plataformas).split(","):
        token = token.strip()
        if token and token != "-" and token.lower() != "unknown":
            filas.append([fila.IdVisita, token])
puente = pd.DataFrame(filas, columns=["IdVisita", "Plataforma"]).drop_duplicates()
print("Asociaciones:", len(puente))

## 25. Leer el catálogo de permisos
El único catálogo externo es `Catalogos/usuarios.txt`.

In [ ]:
usuarios = pd.read_csv(CATALOGOS / "usuarios.txt", dtype=str,
                       encoding="utf-8-sig", keep_default_na=False)
if list(usuarios.columns) != ["Usuario", "Rol", "Plataforma"]:
    raise ValueError("El catálogo de permisos cambió de estructura")
if usuarios[["Usuario", "Rol", "Plataforma"]].eq("").any().any():
    raise ValueError("Hay permisos incompletos")
usuarios.groupby("Usuario")["Plataforma"].agg(list)

## 26. Comprobar la integridad
Si falla un control, no se graba ninguna salida.

In [ ]:
assert list(visitas.columns) == FINAL
assert visitas["IdVisita"].is_unique
assert visitas["VisitasRegistro"].sum() == len(visitas)
assert visitas["FechaVisita"].notna().all()
assert puente["IdVisita"].isin(visitas["IdVisita"]).all()
assert not puente.duplicated(["IdVisita", "Plataforma"]).any()
print("Integridad: OK")

## 27. Preparar los controles de carga
El histórico solo cambia si hay datos nuevos; sin nuevos archivos, la ejecución es idempotente.

In [ ]:
control_cargas = pd.DataFrame({"Archivo": [a.name for a in archivos],
    "Estado": ["Nuevo procesado" if a in nuevos else "Ya conocido" for a in archivos]})
CAMBIOS = bool(nuevos) or not RUTA_ESTADO.exists()
print("¿Se requiere exportación?", CAMBIOS)

## 28. Preparar archivos de salida en zona temporal
Escribimos las tablas completas antes de reemplazar los archivos de salida del proyecto. Así evitamos resultados parciales por errores de transformación.

In [ ]:
TEMP = OUTPUT / "_salida_temporal"
if TEMP.exists():
    raise RuntimeError("Existe una exportación temporal anterior; revisar antes de continuar")
if CAMBIOS:
    TEMP.mkdir()

## 29. Guardar las tablas CSV
Los archivos se exportan con el esquema requerido por el modelo Power BI.

In [ ]:
if CAMBIOS:
    visitas.to_csv(TEMP / "CONSOLIDADO.csv", index=False,
                   encoding="utf-8-sig", date_format="%d/%m/%Y %H:%M")


## 30. Guardar dos libros Excel
Los libros incluyen las hojas `CONSOLIDADO`, `usuarios` y `VisitaPlataforma`.

In [ ]:
if CAMBIOS:
    with pd.ExcelWriter(TEMP / "Modelo_Integracion_Reporting.xlsx", engine="openpyxl",
                        datetime_format="dd/mm/yyyy hh:mm") as excel:
        visitas.to_excel(excel, sheet_name="CONSOLIDADO", index=False)
        usuarios.to_excel(excel, sheet_name="usuarios", index=False)
    puente.to_excel(TEMP / "Interacciones_Plataforma.xlsx",
                    engine="openpyxl", sheet_name="VisitaPlataforma", index=False)

## 31. Preparar manifiesto y control incremental
Se confirma el historial **después** de producir los resultados. No se ejecutan dos procesos a la vez.

In [ ]:
if CAMBIOS:
    (TEMP / "archivos_procesados.json").write_text(
        json.dumps(estado_nuevo, indent=2, ensure_ascii=False), encoding="utf-8")

## 32. Publicar resultados
La marca de control evita un reprocesamiento silencioso tras un cierre inesperado. Si aparece, revisar o restaurar conjuntamente las salidas antes de reiniciar.

In [ ]:
if CAMBIOS:
    RUTA_PENDIENTE.write_text("Carga en curso; revisar si se interrumpe", encoding="utf-8")
    try:
        for nombre in ("CONSOLIDADO.csv",
                       "Modelo_Integracion_Reporting.xlsx", "Interacciones_Plataforma.xlsx",
                       "archivos_procesados.json"):
            os.replace(str(TEMP / nombre), str(OUTPUT / nombre))
    except Exception:
        raise RuntimeError("Publicación incompleta: restaurar output antes de reejecutar")
    else:
        RUTA_PENDIENTE.unlink()
        TEMP.rmdir()

## 33. Conciliación final
Los resultados de referencia corresponden a los tres reportes iniciales; con nuevos archivos los totales cambian legítimamente.

In [ ]:
resumen = pd.DataFrame({"Indicador": ["Visitas", "Usuarios únicos", "Aperturas", "Clics",
                                    "Columnas", "Asociaciones", "Plataformas", "IDs repetidos"],
   "Obtenido": [len(visitas), visitas["EmailNormalizado"].nunique(),
                visitas["Opens"].sum(), visitas["Clicks"].sum(),
                len(visitas.columns), len(puente), puente["Plataforma"].nunique(),
                visitas["IdVisita"].duplicated().sum()]})
resumen